In [20]:
text="Hello! This is a Vishu"
lower=text.lower()
print(lower)


hello! this is a vishu


In [21]:
import re
text='Hello THis is a TEJa'
num_clean=re.sub('[^a-z]','',text)
print(num_clean)

elloisisaa


In [22]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords',quiet=True)
stopwords=set(stopwords.words('english'))
text="I'm going to College today"
tokens =[]
for tec in text.split():
  if tec not in stopwords:
    tokens.append(tec)
print(tokens)

["I'm", 'going', 'College', 'today']


In [23]:
import nltk
nltk.download('punkt_tab',quiet=True)
sentence=" I don't love sapm emails ,do you?"
print('basic split:',sentence.split())


basic split: ['I', "don't", 'love', 'sapm', 'emails', ',do', 'you?']


In [24]:
from nltk.tokenize import word_tokenize
tokens=word_tokenize(sentence.lower())
print('NLTK tokenize:',tokens)

NLTK tokenize: ['i', 'do', "n't", 'love', 'sapm', 'emails', ',', 'do', 'you', '?']


In [25]:
import nltk
from nltk.stem import PorterStemmer,WordNetLemmatizer
nltk.download('wordnet',quiet=True)
stemmer=PorterStemmer()
lemmatizer=WordNetLemmatizer()
words=['running','study','caring','happily','was']
for word in words:
  stem=stemmer.stem(word)
  lemm=lemmatizer.lemmatize(word,pos='v')
  print(f'{word:12} |stem:{stem:10} | lemma:{lemm:204}')

running      |stem:run        | lemma:run                                                                                                                                                                                                         
study        |stem:studi      | lemma:study                                                                                                                                                                                                       
caring       |stem:care       | lemma:care                                                                                                                                                                                                        
happily      |stem:happili    | lemma:happily                                                                                                                                                                                                     
was          |stem:wa       

In [26]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer,WordNetLemmatizer
nltk.download(['stopwords','wordnet','punkt'],quiet=True)
stop_words=set(stopwords.words('english'))
lemmatizer=WordNetLemmatizer()
def preprocess(text):
  text=text.lower()
  text=re.sub('[^a-z]',' ',text)
  tokens=text.split()
  filtered_tokens=[]
  for w in tokens:
    if w not in stop_words:
      filtered_tokens.append(w)
  tokens=filtered_tokens
  lemmatized_tokens=[]
  for w in tokens:
      lemmatized_tokens.append(
          lemmatizer.lemmatize(w)
      )
  tokens=lemmatized_tokens
  return ' '.join(tokens)
print(preprocess('i am running to the stores to buy some grocery '))

running store buy grocery


In [27]:
from sklearn.feature_extraction.text import CountVectorizer
emails = [
    'cricket match cricket match cricket',
    'free win money free win money win',
    'cricket match free win money cricket',
]
cv = CountVectorizer()
X = cv.fit_transform(emails).toarray()
print('Vocabulary:', cv.get_feature_names_out())
print('Bow matrix: ')
print(X)

Vocabulary: ['cricket' 'free' 'match' 'money' 'win']
Bow matrix: 
[[3 0 2 0 0]
 [0 2 0 2 3]
 [2 1 1 1 1]]


In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer
emails = [
    'cricket match cricket match cricket',
    'free win money free win money win',
    'cricket match free win money cricket',
]
tfidf = TfidfVectorizer(sublinear_tf=True, max_df=0.95, min_df=1)
X_tfidf = tfidf.fit_transform(emails).toarray()
print('Feature Names:', tfidf.get_feature_names_out())
print('TF-IDF matrix(rounded): ')
import numpy as np
print(np.round(X_tfidf, 3))

Feature Names: ['cricket' 'free' 'match' 'money' 'win']
TF-IDF matrix(rounded): 
[[0.778 0.    0.628 0.    0.   ]
 [0.    0.532 0.    0.532 0.659]
 [0.646 0.382 0.382 0.382 0.382]]


**LAB**

#**Spam Email Classifier**
Build a real spam detector using TF-IDF + Logistic Regression from scratch.

**Steps**


Step1 : Loading and explore the data.

Step2 : Pre-processing data and vectorise.

Step3 : Training the model and evalation.

Step4 : Predcting new messages.

Step5 :


In [32]:
#Step1
import pandas as pd
import numpy as np
!wget -q https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip
!unzip -o smsspamcollection.zip
!mv SMSSpamCollection SMSSpamCollection
df = pd.read_csv('SMSSpamCollection', sep='\t',
header=None, names=['label', 'message'])
# Quick look at the data
print(df.head())
print('\nShape:', df.shape) # (5574, 2)
print('\nClass balance:')
print(df['label'].value_counts()) # ham: 4827, spam: 747
print(f'Spam rate: {747/5574*100:.1f}%') # ~13% spam
# Check a spam example
print('\nSpam example:')
print(df[df['label']=='spam']['message'].iloc[0])
# 'Free entry in 2 a wkly comp to win FA Cup...'

#Step2

import re, nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
nltk.download(['stopwords','wordnet'], quiet=True)
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
def preprocess(text):
  text = text.lower() # lowercase
  text = re.sub(r'[^a-z ]', '', text) # remove punctuation
  tokens = [w for w in text.split() if w not in stop_words] # stopwords
  tokens = [lemmatizer.lemmatize(w) for w in tokens] # lemmatise
  return ' '.join(tokens)
# Apply pre-processing to every message
df['clean'] = df['message'].apply(preprocess)
# Convert labels: spam=1, ham=0
df['target'] = (df['label'] == 'spam').astype(int)
# TF-IDF vectorisation
# max_features=5000 keeps only the top 5000 most informative words
# ngram_range=(1,2) also captures 2-word phrases like 'free win'
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2),
sublinear_tf=True)
X = tfidf.fit_transform(df['clean']) # sparse matrix (5574 x 5000)
y = df['target'].values
# Split: 80% train, 20% test, stratify keeps spam ratio balanced in both
X_tr, X_te, y_tr, y_te = train_test_split(
X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_tr.shape}, Test: {X_te.shape}')

#Step3

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report,
confusion_matrix, accuracy_score)
import seaborn as sns
import matplotlib.pyplot as plt
# Train Logistic Regression
# C=1.0 is regularisation strength (higher = less regularisation)
# max_iter=1000 ensures the model converges
clf = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
clf.fit(X_tr, y_tr) # this is where the model actually 'learns'
# Evaluate on test set (data the model has never seen)
y_pred = clf.predict(X_te)
print('Accuracy:', accuracy_score(y_te, y_pred))
# Accuracy: ~0.984 (98.4% correct!)
# Full report: precision, recall, F1
print(classification_report(y_te, y_pred,
target_names=['Ham (genuine)', 'Spam']))
# Confusion matrix
cm = confusion_matrix(y_te, y_pred)
print('Confusion matrix:')
print(cm)
# [[True Ham, False Spam],
# [Missed Spam, Caught Spam]]

#Step4
def predict_spam(message):
  '''Predict if a new message is spam or ham.'''
  cleaned = preprocess(message) # same pipeline as training
  vector = tfidf.transform([cleaned]) # transform (not fit_transform!)
  pred = clf.predict(vector)[0]
  prob = clf.predict_proba(vector)[0]
  label = 'SPAM' if pred==1 else 'HAM'
  conf = prob[pred]
  return f'{label} (confidence: {conf:.1%})'
# Test on new messages
messages = [
'Congratulations! You won a free iPhone. Click here to claim now!',
'Hey, are we still meeting at 6pm for dinner?',
'URGENT: Your bank account has been suspended. Verify now!',
'Can you pick up milk on the way home?',
]
for msg in messages:
  print(f'{predict_spam(msg):30} | {msg[:55]}...')
# SPAM (confidence: 99.8%) | Congratulations! You won a free iPhone...
# HAM (confidence: 99.1%) | Hey, are we still meeting at 6pm...
# SPAM (confidence: 98.5%) | URGENT: Your bank account has been...
# HAM (confidence: 97.3%) | Can you pick up milk on the way home?


Archive:  smsspamcollection.zip
  inflating: SMSSpamCollection       
  inflating: readme                  
mv: 'SMSSpamCollection' and 'SMSSpamCollection' are the same file
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...

Shape: (5572, 2)

Class balance:
label
ham     4825
spam     747
Name: count, dtype: int64
Spam rate: 13.4%

Spam example:
Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
Train: (4457, 5000), Test: (1115, 5000)
Accuracy: 0.9659192825112107
               precision    recall  f1-score   support

Ham (genuine)       0.96      1.00      0.98       966
         Spa

# NLP Interview Questions and Answers

==================================================================

Q1. What is Tokenisation and Why Is It Needed?

Answer:

Tokenisation is the process of splitting raw text into smaller units called
tokens.

Tokens may be:

• Words

• Sub-words

• Characters

• Sentences

Example:

Text:
    "I love NLP"

Tokens:
    ["I", "love", "NLP"]

Why is tokenisation needed?

Machine Learning models cannot understand raw text directly.

They require text to be converted into structured units before
creating numerical representations.

Benefits:

• Builds vocabulary mappings

• Enables word frequency counting

• Forms the foundation for text preprocessing

• Converts unstructured text into model-friendly format

------------------------------------------------------------------

Q2. What is the Difference Between Stemming and Lemmatization?
Give an Example.

Answer:

Both techniques reduce words to their base form.

--------------------------------------------------

Stemming

• Uses simple rule-based chopping

• Removes prefixes or suffixes

• Does not check dictionary validity

• Faster

Examples:

running → run

playing → play

better → bett

Notice that "bett" is not a valid English word.

--------------------------------------------------

Lemmatization

• Uses vocabulary and grammar rules

• Considers context and part of speech

• Produces valid dictionary words

• Slower but more accurate

Examples:

running → run

playing → play

better → good

--------------------------------------------------

Key Difference

Stemming:
    Fast but may create invalid words.

Lemmatization:
    Slower but produces meaningful root words.

------------------------------------------------------------------

Q3. Why Do We Remove Stopwords?
Can Removing Them Ever Be Harmful?

Answer:

Stopwords are common words that appear frequently in text.

Examples:

• the

• is

• am

• are

• and

• of

• in

These words usually carry little information and increase noise.

Benefits of removing stopwords:

• Reduces vocabulary size

• Improves processing speed

• Removes irrelevant information

• Improves model efficiency

--------------------------------------------------

Can it be harmful?

Yes.

Example:

Sentence:
    "This movie is not good."

If "not" is removed:

    "This movie good"

The sentiment completely changes.

Therefore:

For Sentiment Analysis, Chatbots,

Question Answering,

and Language Understanding tasks,

stopword removal
must be done carefully.

------------------------------------------------------------------

Q4. What Is TF-IDF and Why Is It Better Than Bag of Words?

Answer:

TF-IDF stands for:

TF = Term Frequency

IDF = Inverse Document Frequency

Formula:

:contentReference[oaicite:0]{index=0}

--------------------------------------------------

Term Frequency (TF)

Measures how often a word appears in a document.

Example:

Document:
    "cat cat dog"

TF(cat) = 2

TF(dog) = 1

--------------------------------------------------

Inverse Document Frequency (IDF)

Measures how rare a word is across all documents.

Common words get lower scores.

Rare words get higher scores.

--------------------------------------------------

Why TF-IDF is Better than Bag of Words

Bag of Words:

• Uses raw word counts

• Treats all words equally

TF-IDF:

• Rewards important words

• Penalizes common words

• Produces more meaningful features

Example:

Word:
    "the"

Appears in every document.

Bag of Words:
    High importance

TF-IDF:
    Low importance

Thus TF-IDF provides more informative representations.

------------------------------------------------------------------

Q5. What Evaluation Metric Would You Use for a Spam Classifier and Why?

Answer:

Accuracy alone can be misleading when classes are imbalanced.

Example:

Dataset:

87% Ham (Normal Emails)
13% Spam Emails

A model predicting everything as Ham gives:

Accuracy = 87%

But it detects zero spam.

--------------------------------------------------

Better Metrics

1. Precision

Measures:

    Of emails predicted as spam,
    how many were actually spam?

Formula:

:contentReference[oaicite:1]{index=1}

--------------------------------------------------

2. Recall

Measures:

    Of all actual spam emails,
    how many were detected?

Formula:

:contentReference[oaicite:2]{index=2}

--------------------------------------------------

3. F1 Score

Balances Precision and Recall.

Formula:

:contentReference[oaicite:3]{index=3}

--------------------------------------------------

Production Preference

Spam Detection usually prioritizes:

High Recall

Reason:

Missing spam emails is generally worse than accidentally
flagging a few genuine emails.

------------------------------------------------------------------

Q6. How Does Logistic Regression Classify Text After TF-IDF Vectorisation?

Answer:

Step 1: Convert Text to TF-IDF Features

Example:

Email:
    "Win free iPhone now"

TF-IDF converts the email into a numeric vector.

Example:

[0.82, 0.67, 0.91, 0.54]

--------------------------------------------------

Step 2: Learn Feature Weights

Logistic Regression learns a weight for each word.

Example:

"free"   → +2.4

"win"    → +3.1

"offer"  → +1.8

"meeting"→ -2.0

Positive weights indicate spam-like words.

Negative weights indicate normal-email words.

--------------------------------------------------

Step 3: Calculate Weighted Sum

The model computes:

z = w₁x₁ + w₂x₂ + ... + b

--------------------------------------------------

Step 4: Apply Sigmoid Function

:contentReference[oaicite:4]{index=4}

The sigmoid converts the score into a probability
between 0 and 1.

Example:

P(spam) = 0.92

--------------------------------------------------

Step 5: Make Prediction

If:

P(spam) > 0.5

→ Spam

Else

→ Ham (Normal Email)

--------------------------------------------------

Summary:

Text

  ↓

Tokenisation

  ↓

Preprocessing

  ↓

TF-IDF Vectorisation

  ↓

Logistic Regression

  ↓

Weighted Sum

  ↓

Sigmoid Function

  ↓

Spam Probability

  ↓
  
Final Classification
